#企业风险分析Pipeline
###医药生物行业信用风险模型 —— 方案：ST事件预测
**流程：** 筛选医药生物行业 → 爬虫A股上市公司2020-2025年财务数据，ST数据，历年年报 → 提取MDA文本 → NLP-FinBERT情感分析 → 写回长表 → 逻辑回归，XGBOOST

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install akshare -q

import akshare as ak
import pandas as pd
import time

base_path = '/content/drive/MyDrive/credit_risk_project'
YEARS = ['20201231', '20211231', '20221231', '20231231', '20241231', '20251231']

In [ ]:
#test-成功的接口
ak.stock_lrb_em(date="20231231")   # 利润表
ak.stock_zcfz_em(date="20231231")  # 资产负债表
ak.stock_xjll_em(date="20231231")  # 现金流量表

In [ ]:
import os

base_path = '/content/drive/MyDrive/credit_risk_project'

os.makedirs(os.path.join(base_path, 'Industry_classification'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'Company_Basic_Information'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'Income_Statement'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'Balance_Sheet'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'Cash_Flow_Statement'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'annual_report_pdf'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'finance_data'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'logs'), exist_ok=True)

print(f"Created directory structure under {base_path}")

Created directory structure under /content/drive/MyDrive/credit_risk_project


In [ ]:
# Table 1：利润表
frames = []
for date in YEARS:
    try:
        df = ak.stock_lrb_em(date=date)
        df.insert(0, '报告期', date[:4] + '-' + date[4:6] + '-' + date[6:])  # 格式化为 YYYY-MM-DD
        frames.append(df)
        print(f'✅ 利润表 {date}: {len(df)} 行')
    except Exception as e:
        print(f'❌ 利润表 {date}: {e}')
    time.sleep(1)

income = pd.concat(frames, ignore_index=True)

# 把股票代码移到第一列
cols = ['股票代码', '报告期'] + [c for c in income.columns if c not in ('股票代码', '报告期')]
income = income[cols]

path = f'{BASE}/Income_Statement/Income_Statement.csv'
income.to_csv(path, index=False, encoding='utf-8-sig')
print(f'\n☑️ 利润表保存完毕：{len(income)} 行 × {len(income.columns)} 列')
print(income.head(3))

✅ 利润表 20201231: 5176 行


✅ 利润表 20211231: 5183 行


❌ 利润表 20221231: Response ended prematurely


✅ 利润表 20231231: 5217 行


✅ 利润表 20241231: 5213 行


✅ 利润表 20251231: 5210 行

☑️ 利润表保存完毕：25999 行 × 16 列
     股票代码         报告期  序号  股票简称           净利润       净利润同比         营业总收入  \
0  688189  2020-12-31   1  南新制药  1.279720e+08   39.900000  1.029142e+09   
1  688076  2020-12-31   2  ST诺泰  1.234416e+08  153.940197  5.668725e+08   
2  603690  2020-12-31   3  至纯科技  2.605997e+08  136.360000  1.397056e+09   

     营业总收入同比    营业总支出-营业支出    营业总支出-销售费用    营业总支出-管理费用   营业总支出-财务费用  \
0   1.470984  1.048808e+08  6.035256e+08  5.975086e+07   5888528.16   
1  52.577609  2.371562e+08  1.154802e+07  1.132829e+08  19589369.19   
2  41.626178  8.830838e+08  5.441784e+07  1.376165e+08  76332699.70   

    营业总支出-营业总支出          营业利润          利润总额        公告日期  
0  8.817037e+08  1.319246e+08  1.343974e+08  2026-04-30  
1  4.457577e+08  1.438154e+08  1.465610e+08  2026-04-30  
2  1.256576e+09  2.968769e+08  2.981806e+08  2026-04-30  


In [ ]:
import akshare as ak
import pandas as pd
import time

BASE = '/content/drive/MyDrive/credit_risk_project'

# 单独补拉2022年利润表
for attempt in range(5):
    try:
        df = ak.stock_lrb_em(date='20221231')
        df.insert(0, '报告期', '2022-12-31')
        print(f'✅ 成功: {len(df)} 行')
        break
    except Exception as e:
        print(f'第{attempt+1}次失败: {e}')
        time.sleep(3)

# 读取已有的利润表，追加2022年数据，重新保存
path = f'{BASE}/Income_Statement/Income_Statement.csv'
existing = pd.read_csv(path, dtype={'股票代码': str})
merged = pd.concat([existing, df], ignore_index=True)

# 按报告期排序
merged = merged.sort_values(['股票代码', '报告期']).reset_index(drop=True)
merged.to_csv(path, index=False, encoding='utf-8-sig')
print(f'☑️ 补充完毕，利润表现在共 {len(merged)} 行')

✅ 成功: 5218 行
☑️ 补充完毕，利润表现在共 31217 行


In [ ]:
# Table 2：资产负债表
frames = []
for date in YEARS:
    try:
        df = ak.stock_zcfz_em(date=date)
        df.insert(0, '报告期', date[:4] + '-' + date[4:6] + '-' + date[6:])
        frames.append(df)
        print(f'✅ 资产负债表 {date}: {len(df)} 行')
    except Exception as e:
        print(f'❌ 资产负债表 {date}: {e}')
    time.sleep(1)

balance = pd.concat(frames, ignore_index=True)

cols = ['股票代码', '报告期'] + [c for c in balance.columns if c not in ('股票代码', '报告期')]
balance = balance[cols]

path = f'{BASE}/Balance_Sheet/Balance_Sheet.csv'
balance.to_csv(path, index=False, encoding='utf-8-sig')
print(f'\n☑️ 资产负债表保存完毕：{len(balance)} 行 × {len(balance.columns)} 列')
print(balance.head(3))

✅ 资产负债表 20201231: 5176 行


✅ 资产负债表 20211231: 5183 行


✅ 资产负债表 20221231: 5218 行


✅ 资产负债表 20231231: 5217 行


✅ 资产负债表 20241231: 5213 行


✅ 资产负债表 20251231: 5210 行

☑️ 资产负债表保存完毕：31217 行 × 16 列
     股票代码         报告期  序号  股票简称       资产-货币资金       资产-应收账款         资产-存货  \
0  688189  2020-12-31   1  南新制药  2.993510e+08  6.399733e+08  5.333689e+07   
1  688076  2020-12-31   2  ST诺泰  6.971188e+07  1.074225e+08  1.849953e+08   
2  603690  2020-12-31   3  至纯科技  1.502515e+09  9.802368e+08  7.944666e+08   

         资产-总资产    资产-总资产同比       负债-应付账款  负债-预收账款        负债-总负债   负债-总负债同比  \
0  2.086019e+09  154.791094  2.020768e+07      NaN  4.668678e+08   1.173817   
1  1.395573e+09   12.138214  1.054431e+08      NaN  4.295486e+08   5.921115   
2  5.956663e+09   82.882038  3.454086e+08      NaN  2.795942e+09  59.201229   

       资产负债率        股东权益合计        公告日期  
0  22.380803  1.619151e+09  2026-04-30  
1  30.779360  9.660249e+08  2026-04-30  
2  46.938067  3.160720e+09  2026-04-30  


In [ ]:
# Table 3：现金流量表
frames = []
for date in YEARS:
    try:
        df = ak.stock_xjll_em(date=date)
        df.insert(0, '报告期', date[:4] + '-' + date[4:6] + '-' + date[6:])
        frames.append(df)
        print(f'✅ 现金流量表 {date}: {len(df)} 行')
    except Exception as e:
        print(f'❌ 现金流量表 {date}: {e}')
    time.sleep(1)

cashflow = pd.concat(frames, ignore_index=True)

cols = ['股票代码', '报告期'] + [c for c in cashflow.columns if c not in ('股票代码', '报告期')]
cashflow = cashflow[cols]

path = f'{BASE}/Cash_Flow_Statement/Cash_Flow_Statement.csv'
cashflow.to_csv(path, index=False, encoding='utf-8-sig')
print(f'\n☑️ 现金流量表保存完毕：{len(cashflow)} 行 × {len(cashflow.columns)} 列')
print(cashflow.head(3))

✅ 现金流量表 20201231: 5176 行


✅ 现金流量表 20211231: 5183 行


✅ 现金流量表 20221231: 5218 行


✅ 现金流量表 20231231: 5217 行


✅ 现金流量表 20241231: 5213 行


✅ 现金流量表 20251231: 5210 行

☑️ 现金流量表保存完毕：31217 行 × 13 列
     股票代码         报告期  序号   股票简称     净现金流-净现金流   净现金流-同比增长  经营性现金流-现金流量净额  \
0  688076  2020-12-31   1   ST诺泰  9.975282e+06  107.607773   1.413813e+08   
1  603690  2020-12-31   2   至纯科技  1.028059e+09  203.605810  -2.809275e+08   
2  600238  2020-12-31   3  *ST椰岛  1.088887e+08  204.081471   1.140410e+08   

   经营性现金流-净现金流占比  投资性现金流-现金流量净额  投资性现金流-净现金流占比  融资性现金流-现金流量净额  融资性现金流-净现金流占比  \
0    1417.316697  -1.631751e+08   -1635.794648   3.950240e+07     396.002859   
1     -27.325999  -9.090727e+08     -88.426090   2.218380e+09     215.783292   
2     104.731734   2.879446e+07      26.443936  -3.394678e+07     -31.175670   

         公告日期  
0  2026-04-30  
1  2026-04-30  
2  2026-04-30  


In [ ]:
# Table 4：申万行业分类
# 获取申万一级行业列表（31个）
sw_index = ak.sw_index_first_info()
sw_index = sw_index[['行业代码', '行业名称']].copy()
print(f'申万一级行业数: {len(sw_index)}')

frames = []
for _, row in sw_index.iterrows():
    raw_code = row['行业代码']          # e.g. '801010.SI'
    name     = row['行业名称']
    clean_code = raw_code.split('.')[0]  # '801010'
    try:
        cons = ak.index_component_sw(symbol=clean_code)
        cons = cons[['证券代码', '证券名称']].copy()
        cons.columns = ['股票代码', '股票简称']
        cons['行业代码']     = raw_code
        cons['申万一级行业'] = name
        frames.append(cons)
        print(f'✅ {name}: {len(cons)} 只')
    except Exception as e:
        print(f'❌ {name}: {e}')
    time.sleep(0.5)

industry = pd.concat(frames, ignore_index=True)
industry = industry[['股票代码', '股票简称', '行业代码', '申万一级行业']]

path = f'{BASE}/Industry_classification/Industry_classification.csv'
industry.to_csv(path, index=False, encoding='utf-8-sig')
print(f'\n☑️ 行业分类保存完毕：{len(industry)} 行 × {len(industry.columns)} 列')
print(industry.head(3))

申万一级行业数: 31
✅ 农林牧渔: 104 只
✅ 基础化工: 410 只
✅ 钢铁: 44 只
✅ 有色金属: 142 只
✅ 电子: 481 只
✅ 汽车: 286 只
✅ 家用电器: 94 只
✅ 食品饮料: 122 只
✅ 纺织服饰: 106 只
✅ 轻工制造: 158 只
✅ 医药生物: 479 只
✅ 公用事业: 131 只
✅ 交通运输: 126 只
✅ 房地产: 99 只
✅ 商贸零售: 99 只
✅ 社会服务: 80 只
✅ 银行: 42 只
✅ 非银金融: 81 只
✅ 综合: 15 只
✅ 建筑材料: 72 只
✅ 建筑装饰: 156 只
✅ 电力设备: 369 只
✅ 机械设备: 534 只
✅ 国防军工: 138 只
✅ 计算机: 334 只
✅ 传媒: 130 只
✅ 通信: 123 只
✅ 煤炭: 37 只
✅ 石油石化: 47 只
✅ 环保: 133 只
✅ 美容护理: 29 只

☑️ 行业分类保存完毕：5201 行 × 4 列
     股票代码  股票简称       行业代码 申万一级行业
0  000505  京粮控股  801010.SI   农林牧渔
1  000592  平潭发展  801010.SI   农林牧渔
2  000702  正虹科技  801010.SI   农林牧渔


In [ ]:
import pandas as pd

Balance_sheet = pd.read_csv('/content/drive/MyDrive/credit_risk_project/Balance_Sheet/Balance_Sheet.csv')
Balance_sheet.head(5)

,股票代码,报告期,序号,股票简称,资产-货币资金,资产-应收账款,资产-存货,资产-总资产,资产-总资产同比,负债-应付账款,负债-预收账款,负债-总负债,负债-总负债同比,资产负债率,股东权益合计,公告日期
0,688189,2020-12-31,1,南新制药,2.993510e+08,6.399733e+08,5.333689e+07,2.086019e+09,154.791094,2.020768e+07,NaN,4.668678e+08,1.173817,22.380803,1.619151e+09,2026-04-30
1,688076,2020-12-31,2,ST诺泰,6.971188e+07,1.074225e+08,1.849953e+08,1.395573e+09,12.138214,1.054431e+08,NaN,4.295486e+08,5.921115,30.779360,9.660249e+08,2026-04-30
2,603690,2020-12-31,3,至纯科技,1.502515e+09,9.802368e+08,7.944666e+08,5.956663e+09,82.882038,3.454086e+08,NaN,2.795942e+09,59.201229,46.938067,3.160720e+09,2026-04-30
3,600238,2020-12-31,4,*ST椰岛,2.200678e+08,8.964699e+07,2.363382e+08,1.181256e+09,-0.030993,4.276498e+07,NaN,6.237919e+08,-3.713844,52.807521,5.574639e+08,2026-04-30
4,600076,2020-12-31,5,康欣新材,3.200524e+08,2.990415e+08,3.261462e+09,7.140218e+09,8.371445,1.968725e+08,NaN,3.142199e+09,29.528655,44.007049,3.998019e+09,2026-04-30


In [ ]:
Cash_Flow_Statement = pd.read_csv('/content/drive/MyDrive/credit_risk_project/Cash_Flow_Statement/Cash_Flow_Statement.csv')
Cash_Flow_Statement.head(5)

,股票代码,报告期,序号,股票简称,净现金流-净现金流,净现金流-同比增长,经营性现金流-现金流量净额,经营性现金流-净现金流占比,投资性现金流-现金流量净额,投资性现金流-净现金流占比,融资性现金流-现金流量净额,融资性现金流-净现金流占比,公告日期
0,688076,2020-12-31,1,ST诺泰,9.975282e+06,107.607773,1.413813e+08,1417.316697,-1.631751e+08,-1635.794648,3.950240e+07,396.002859,2026-04-30
1,603690,2020-12-31,2,至纯科技,1.028059e+09,203.605810,-2.809275e+08,-27.325999,-9.090727e+08,-88.426090,2.218380e+09,215.783292,2026-04-30
2,600238,2020-12-31,3,*ST椰岛,1.088887e+08,204.081471,1.140410e+08,104.731734,2.879446e+07,26.443936,-3.394678e+07,-31.175670,2026-04-30
3,600725,2020-12-31,4,云维股份,2.827784e+07,-15.815266,2.001569e+07,70.782241,6.707199e+05,2.371893,7.591430e+06,26.845866,2026-04-29
4,600082,2020-12-31,5,ST海泰,1.282136e+08,127.687889,2.278208e+07,17.768850,9.972861e+07,77.783178,5.702906e+06,4.447973,2026-04-29


In [ ]:
Income_Statement = pd.read_csv('/content/drive/MyDrive/credit_risk_project/Income_Statement/Income_Statement.csv')
Income_Statement.head(5)

,股票代码,报告期,序号,股票简称,净利润,净利润同比,营业总收入,营业总收入同比,营业总支出-营业支出,营业总支出-销售费用,营业总支出-管理费用,营业总支出-财务费用,营业总支出-营业总支出,营业利润,利润总额,公告日期
0,1,2020-12-31,5098,平安银行,2.892800e+10,2.6,1.535420e+11,11.296192,NaN,NaN,4.469000e+10,NaN,1.166330e+11,3.690900e+10,3.675400e+10,2022-03-10
1,1,2021-12-31,5129,平安银行,3.633600e+10,25.6,1.693830e+11,10.317047,NaN,NaN,4.793700e+10,NaN,1.233980e+11,4.598500e+10,4.587900e+10,2023-03-09
2,1,2022-12-31,5154,平安银行,4.551600e+10,25.3,1.798950e+11,6.206054,NaN,NaN,4.938700e+10,NaN,1.224200e+11,5.747500e+10,5.725300e+10,2024-03-15
3,1,2023-12-31,5139,平安银行,4.645500e+10,2.1,1.646990e+11,-8.447150,NaN,NaN,4.595900e+10,NaN,1.067710e+11,5.792800e+10,5.771800e+10,2025-03-15
4,1,2024-12-31,5034,平安银行,4.450800e+10,-4.2,1.466950e+11,-10.931457,NaN,NaN,4.058200e+10,NaN,9.148900e+10,5.520600e+10,5.473800e+10,2026-03-21


In [ ]:
Industry_classification = pd.read_csv('/content/drive/MyDrive/credit_risk_project/Industry_classification/Industry_classification.csv')
Industry_classification.head(5)

,股票代码,股票简称,行业代码,申万一级行业
0,505,京粮控股,801010.SI,农林牧渔
1,592,平潭发展,801010.SI,农林牧渔
2,702,正虹科技,801010.SI,农林牧渔
3,713,国投丰乐,801010.SI,农林牧渔
4,735,罗牛山,801010.SI,农林牧渔


In [ ]:
BASE = '/content/drive/MyDrive/credit_risk_project'

income   = pd.read_csv(f'{BASE}/Income_Statement/Income_Statement.csv', dtype={'股票代码': str}, nrows=0)
balance  = pd.read_csv(f'{BASE}/Balance_Sheet/Balance_Sheet.csv', dtype={'股票代码': str}, nrows=0)
cashflow = pd.read_csv(f'{BASE}/Cash_Flow_Statement/Cash_Flow_Statement.csv', dtype={'股票代码': str}, nrows=0)
industry = pd.read_csv(f'{BASE}/Industry_classification/Industry_classification.csv', dtype={'股票代码': str}, nrows=0)

for name, df in [('利润表', income), ('资产负债表', balance), ('现金流量表', cashflow), ('行业', industry)]:
    print(f'\n── {name} ({len(df.columns)}列) ──')
    for i, col in enumerate(df.columns):
        print(f'  {i:3d}  {col}')


── 利润表 (16列) ──
    0  股票代码
    1  报告期
    2  序号
    3  股票简称
    4  净利润
    5  净利润同比
    6  营业总收入
    7  营业总收入同比
    8  营业总支出-营业支出
    9  营业总支出-销售费用
   10  营业总支出-管理费用
   11  营业总支出-财务费用
   12  营业总支出-营业总支出
   13  营业利润
   14  利润总额
   15  公告日期

── 资产负债表 (16列) ──
    0  股票代码
    1  报告期
    2  序号
    3  股票简称
    4  资产-货币资金
    5  资产-应收账款
    6  资产-存货
    7  资产-总资产
    8  资产-总资产同比
    9  负债-应付账款
   10  负债-预收账款
   11  负债-总负债
   12  负债-总负债同比
   13  资产负债率
   14  股东权益合计
   15  公告日期

── 现金流量表 (13列) ──
    0  股票代码
    1  报告期
    2  序号
    3  股票简称
    4  净现金流-净现金流
    5  净现金流-同比增长
    6  经营性现金流-现金流量净额
    7  经营性现金流-净现金流占比
    8  投资性现金流-现金流量净额
    9  投资性现金流-净现金流占比
   10  融资性现金流-现金流量净额
   11  融资性现金流-净现金流占比
   12  公告日期

── 行业 (4列) ──
    0  股票代码
    1  股票简称
    2  行业代码
    3  申万一级行业


In [ ]:
BASE = '/content/drive/MyDrive/credit_risk_project'

# 读取，股票代码全部强制str
income   = pd.read_csv(f'{BASE}/Income_Statement/Income_Statement.csv',       dtype={'股票代码': str})
balance  = pd.read_csv(f'{BASE}/Balance_Sheet/Balance_Sheet.csv',             dtype={'股票代码': str})
cashflow = pd.read_csv(f'{BASE}/Cash_Flow_Statement/Cash_Flow_Statement.csv', dtype={'股票代码': str})
industry = pd.read_csv(f'{BASE}/Industry_classification/Industry_classification.csv', dtype={'股票代码': str})

# 补全股票代码前导零到6位
for df in [income, balance, cashflow, industry]:
    df['股票代码'] = df['股票代码'].str.zfill(6)

# 选列
income   = income[['股票代码', '报告期', '股票简称', '净利润', '营业总收入', '营业利润', '利润总额']]
balance  = balance[['股票代码', '报告期', '资产-货币资金', '资产-总资产', '负债-总负债', '资产负债率', '股东权益合计']]
cashflow = cashflow[['股票代码', '报告期', '净现金流-净现金流', '经营性现金流-现金流量净额']]
industry = industry[['股票代码', '申万一级行业']]

# 排序
for df in [income, balance, cashflow]:
    df.sort_values(['股票代码', '报告期'], inplace=True)
    df.reset_index(drop=True, inplace=True)

# 三张财务表 outer join
master = income.merge(balance,  on=['股票代码', '报告期'], how='outer')
master = master.merge(cashflow, on=['股票代码', '报告期'], how='outer')

# 挂行业标签
master = master.merge(industry, on='股票代码', how='left')

# 剔除金融行业
master = master[~master['申万一级行业'].isin(['银行', '非银金融', '综合'])]

# 整理列顺序
cols = ['股票代码', '股票简称', '报告期', '申万一级行业',
        '营业总收入', '营业利润', '利润总额', '净利润',
        '资产-货币资金', '资产-总资产', '负债-总负债', '资产负债率', '股东权益合计',
        '净现金流-净现金流', '经营性现金流-现金流量净额']
master = master[cols]

# 按股票代码+报告期排序
master.sort_values(['股票代码', '报告期'], inplace=True)
master.reset_index(drop=True, inplace=True)

# 保存
out_path = f'{BASE}/finance_data/master_table.csv'
master.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'☑️ 合并完毕：{len(master)} 行 × {len(master.columns)} 列')
print(f'股票数：{master["股票代码"].nunique()}')
print(f'报告期：{sorted(master["报告期"].unique())}')
print(f'行业分布：\n{master["申万一级行业"].value_counts()}')
print()
print(master.head(6).to_string())

☑️ 合并完毕：30389 行 × 15 列
股票数：5080
报告期：['2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
行业分布：
申万一级行业
机械设备    3196
医药生物    2872
电子      2865
基础化工    2456
电力设备    2207
计算机     2002
汽车      1707
轻工制造     946
建筑装饰     936
有色金属     843
国防军工     824
环保       798
公用事业     786
传媒       780
交通运输     756
通信       736
食品饮料     732
纺织服饰     633
农林牧渔     624
商贸零售     594
房地产      594
家用电器     564
社会服务     480
建筑材料     430
石油石化     282
钢铁       264
煤炭       222
美容护理     174
Name: count, dtype: int64

     股票代码 股票简称         报告期 申万一级行业         营业总收入          营业利润          利润总额           净利润       资产-货币资金        资产-总资产        负债-总负债      资产负债率        股东权益合计     净现金流-净现金流  经营性现金流-现金流量净额
0  000002  万科A  2020-12-31    房地产  4.191117e+11  7.995864e+10  7.967575e+10  4.151554e+10  1.952307e+11  1.869177e+12  1.519333e+12  81.283503  3.498445e+11  2.592373e+10   5.318802e+10
1  000002  万科A  2021-12-31    房地产  4.527978e+11  5.253100e+10  5.222263e+10  2.252403e+10  1.493524e+11

提取5张表格

Industry Classification
Company Basic Information
Income Statement
Balance Sheet
Cash Flow Statement

In [ ]:
pip install akshare

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 62.7 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=5c0a8cc85756620fdcab2875990cffb6ddd086a7e3f82b1b168f2f46ad04fc8d
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath


In [ ]:
# 测试巨潮年报接口
df = ak.stock_zh_a_disclosure_report_cninfo(
    symbol="000001",
    market="沪深京",
    category="年报",
    start_date="20200101",
    end_date="20251231"
)
print(df.columns.tolist())
print(df.head())

['代码', '简称', '公告标题', '公告时间', '公告链接']
       代码    简称         公告标题        公告时间  \
0  000001  平安银行  2024年年度报告摘要  2025-03-15   
1  000001  平安银行    2024年年度报告  2025-03-15   
2  000001  平安银行    2023年年度报告  2024-03-15   
3  000001  平安银行  2023年年度报告摘要  2024-03-15   
4  000001  平安银行  2022年年度报告摘要  2023-03-09   

                                                公告链接  
0  http://www.cninfo.com.cn/new/disclosure/detail...  
1  http://www.cninfo.com.cn/new/disclosure/detail...  
2  http://www.cninfo.com.cn/new/disclosure/detail...  
3  http://www.cninfo.com.cn/new/disclosure/detail...  
4  http://www.cninfo.com.cn/new/disclosure/detail...  


In [ ]:
# 打印完整链接
pd.set_option('display.max_colwidth', None)
print(df[["公告标题", "公告链接"]].to_string())

           公告标题                                                                                                                                     公告链接
0   2024年年度报告摘要  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&announcementId=1222806509&orgId=gssz0000001&announcementTime=2025-03-15
1     2024年年度报告  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&announcementId=1222806505&orgId=gssz0000001&announcementTime=2025-03-15
2     2023年年度报告  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&announcementId=1219306493&orgId=gssz0000001&announcementTime=2024-03-15
3   2023年年度报告摘要  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&announcementId=1219306483&orgId=gssz0000001&announcementTime=2024-03-15
4   2022年年度报告摘要  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&announcementId=1216072955&orgId=gssz0000001&announcementTime=2023-03-09
5     2022年年度报告  http://www.cninfo.com.cn/new/disclosure/detail?stockCode=000001&a

In [ ]:
!pip install pdfplumber -q
import requests
import pdfplumber
import io

# 拼PDF直链
announcement_id = "1222806505"
date = "2025-03-15"
pdf_url = f"http://static.cninfo.com.cn/finalpage/{date}/{announcement_id}.PDF"

# 直接读取不保存
response = requests.get(pdf_url)
pdf_bytes = io.BytesIO(response.content)

# 用pdfplumber读取
with pdfplumber.open(pdf_bytes) as pdf:
    print(f"总页数：{len(pdf.pages)}")
    # 读前3页看看
    for i in range(3):
        print(f"\n--- 第{i+1}页 ---")
        print(pdf.pages[i].extract_text()[:200])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 93.9 MB/s eta 0:00:00
总页数：296

--- 第1页 ---
平安银行股份有限公司
年年度报告
2024

--- 第2页 ---
平安银行股份有限公司
重要提示
2024年年度报告
重要提示
1、本行董事会、监事会及董事、监事、高级管理人员保证年度报告内容的真实、准确、完整，
不存在虚假记载、误导性陈述或重大遗漏，并承担个别和连带的法律责任。
2、本行董事长谢永林、行长冀光恒、副行长兼首席财务官项有志、会计机构负责人郁辰声明：
保证本年度报告中财务报告的真实、准确、完整。
3、本行第十二届董事会第三十六次会议审议了2024年

--- 第3页 ---
平安银行股份有限公司
目 录
2024年年度报告
目 录
重要提示.........................................................................................................................................................1
目 录.........


In [ ]:
# 先测试MDA提取，看看平安银行年报的章节标题格式
!pip install pdfplumber
import requests
import pdfplumber
import io
import re

announcement_id = "1222806505"
date = "2025-03-15"
pdf_url = f"http://static.cninfo.com.cn/finalpage/{date}/{announcement_id}.PDF"

response = requests.get(pdf_url)
pdf_bytes = io.BytesIO(response.content)

with pdfplumber.open(pdf_bytes) as pdf:
    full_text = ""
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

# 找所有可能的章节标题（含"讨论"或"分析"或"MDA"的行）
lines = full_text.split('\n')
for i, line in enumerate(lines):
    if any(k in line for k in ['讨论与分析', '经营情况', 'MDA', '管理层']):
        print(f"行{i}: {line.strip()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.6 MB/s eta 0:00:00
行37: 第三章 管理层讨论与分析.......................................................................................................................23
行38: 3.1 总体经营情况..................................................................................................................................23
行40: 3.3 主要业务讨论与分析......................................................................................................................40
行724: 第三章 管理层讨论与分析
行726: 第三章 管理层讨论与分析
行727: 3.1 总体经营情况
行750: 第三章 管理层讨论与分析
行792: 第三章 管理层讨论与分析
行835: 第三章 管理层讨论与分析
行877: 第三章 管理层讨论与分析
行915: 第三章 管理层讨论与分析
行954: 第三章 管理层讨论与分析
行995: 第三章 管理层

In [ ]:
# 测试：按页码定位MDA，而不是按行

announcement_id = "1222806505"
date = "2025-03-15"
pdf_url = f"http://static.cninfo.com.cn/finalpage/{date}/{announcement_id}.PDF"

response = requests.get(pdf_url, timeout=60)
pdf_bytes = io.BytesIO(response.content)

# 第一步：从目录页找MDA页码
with pdfplumber.open(pdf_bytes) as pdf:

    # 目录一般在前10页
    toc_text = ""
    for i in range(10):
        text = pdf.pages[i].extract_text()
        if text:
            toc_text += text + "\n"

    # 从目录找MDA起始页码
    # 格式如："第三章 管理层讨论与分析........23"
    pattern = r'管理层讨论与分析[\.·\s]+(\d+)'
    match = re.search(pattern, toc_text)

    if match:
        mda_start_page = int(match.group(1)) - 1  # 转为0索引
        print(f"MDA起始页：第{mda_start_page+1}页")

        # 找下一章页码
        pattern2 = r'第四章[\.·\s]+(\d+)'
        match2 = re.search(pattern2, toc_text)
        mda_end_page = int(match2.group(1)) - 1 if match2 else mda_start_page + 80
        print(f"MDA结束页：第{mda_end_page+1}页")

        # 提取MDA文本
        mda_text = ""
        for i in range(mda_start_page, mda_end_page):
            text = pdf.pages[i].extract_text()
            if text:
                mda_text += text + "\n"

        print(f"\nMDA字数：{len(mda_text)}")
        print(f"\n前500字预览：")
        print(mda_text[:500])
    else:
        print("目录未找到MDA章节，打印目录内容：")
        print(toc_text)

MDA起始页：第23页
MDA结束页：第103页

MDA字数：89236

前500字预览：
平安银行股份有限公司
第二章 会计数据和财务指标
2024年年度报告
2.7 主要资产重大变化情况
主要资产重大变化情况
主要资产 重大变化说明
股权资产 报告期内无重大变化
固定资产 报告期内无重大变化
无形资产 报告期内无重大变化
在建工程 报告期内无重大变化
主要境外资产情况
□适用 √不适用
22
平安银行股份有限公司
第三章 管理层讨论与分析
2024年年度报告
第三章 管理层讨论与分析
3.1 总体经营情况
2024年，本行坚持党建引领，坚持以“中国最卓越、全球领先的智能化零售银行”为战略目标，
坚持“零售做强、对公做精、同业做专”战略方针，持续升级零售、对公、资金同业业务经营策略，
全力做好金融“五篇大文章”，持续强化风险管理，深化数字化转型，助推高质量发展。
营收利润同比下降，业务经营保持稳健。2024年，受市场变化、主动调整资产结构等因素的影
响，本集团实现营业收入1,466.95亿元，同比下降10.9%。同时，通过数字化转型驱动经营降本增效，
业务及管理费405.82亿元，同比下降11.7%；加强资产质量管控，加大不良资产清收处置力度，信
用及其他资产减值损失494


In [ ]:
import akshare as ak

for fn_name in ['stock_lrb_em', 'stock_zcfz_em', 'stock_xjll_em']:
    fn = getattr(ak, fn_name)
    try:
        df = fn(date='20231231')
        print(f'✅ {fn_name}: {len(df)} 行, 列: {df.columns.tolist()[:5]}...')
    except Exception as e:
        print(f'❌ {fn_name}: {e}')

✅ stock_lrb_em: 5217 行, 列: ['序号', '股票代码', '股票简称', '净利润', '净利润同比']...


✅ stock_zcfz_em: 5217 行, 列: ['序号', '股票代码', '股票简称', '资产-货币资金', '资产-应收账款']...


✅ stock_xjll_em: 5217 行, 列: ['序号', '股票代码', '股票简称', '净现金流-净现金流', '净现金流-同比增长']...


In [ ]:
#成功的接口
ak.stock_lrb_em(date="20231231")   # 利润表
ak.stock_zcfz_em(date="20231231")  # 资产负债表
ak.stock_xjll_em(date="20231231")  # 现金流量表